# Exploration des Survey Property Maps — DP2 (USDF)

Ce notebook accède aux **survey property maps** de Rubin LSST DP2 dans le butler `dp2_prep`.

Deux types de produits sont disponibles dans la collection `LSSTCam/runs/DRP/v30_0_4/DM-54249` :

1. **`SurveyWidePropertyMapPlot`** — images PNG/PDF des cartes générées par le pipeline (section 3–5).
2. **`consolidated_map`** — objets `HealSparseMap` interrogeables (section 6–10, si disponibles).

**À adapter** : noms de collection et de dataset type selon ce qui est effectivement présent.


Ce notebook est un point d'entrée exploratoire pour accéder aux **survey property maps**  
du jeu de données Rubin LSST **Data Preview 2 (DP2)** stockées dans le butler `dp2_prep` à l'USDF.

L'objectif est de :
1. Lister les collections disponibles dans `dp2_prep` qui pourraient contenir des survey property maps.
2. Identifier les dataset types de type `consolidated_map` disponibles.
3. Récupérer et visualiser quelques cartes à titre d'exploration.

**À adapter** : les noms exacts de collection et de dataset type seront à ajuster  
selon ce qui est effectivement disponible dans le butler au moment de l'exécution.

- **Author:** Sylvie Dagoret-Campagne  
- **Creation date:** 2026-03-11  
- **Environment:** USDF RSP — `LSST` kernel (`lsst_distrib` stack)  
- **Reference DP1 notebook:** `203_1_Survey_property_maps.ipynb`  
- **Reference DP2 access notebook:** `2026-03-07_AccessToDP2.ipynb`
- **DP2 DRP Campaigns on Confluence:** https://rubinobs.atlassian.net/wiki/spaces/DM/pages/661192727/LSSTCam+Intermittent+DRP+Runs
- **DP2-DRP Plots:**:https://usdf-rsp.slac.stanford.edu/plot-navigator

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
from IPython.display import display
from pathlib import Path

# LSST middleware
from lsst.daf.butler import Butler

# skyproj: visualisation de cartes HEALPix / HealSparse
# pip install skyproj  si non disponible dans l'environnement
try:
    import skyproj
    HAS_SKYPROJ = True
    print("skyproj disponible")
except ImportError:
    HAS_SKYPROJ = False
    print("skyproj non disponible — les visualisations all-sky seront désactivées")

# healsparse (optionnel — utile pour inspecter les objets HealSparseMap)
try:
    import healsparse
    HAS_HEALSPARSE = True
    print("healsparse disponible")
except ImportError:
    HAS_HEALSPARSE = False
    print("healsparse non disponible")

## 2. Configuration du Butler

Le dépôt `dp2_prep` est utilisé à la place de `dp1`.  
**TODO** : ajuster `COLLECTION_SPMAPS` avec le nom exact de la collection  
qui contient les survey property maps dans DP2.

In [ ]:
# ── Dépôt Butler DP2 à l'USDF ─────────────────────────────────────────────────
REPO = 'dp2_prep'

# ── Collection cible pour les survey property maps ────────────────────────────
# À ajuster selon ce qui est disponible dans dp2_prep :
# Candidats typiques (exemples non validés) :
#   'LSSTCam/runs/DRP/v30_0_4/DM-54249'
#   'dp2-prep/survey-property-maps'   <- spécifique aux SPMs ?
# seule collection qui a skymap LSSTCam/runs/DRP/v30_0_4/DM-54249'  
COLLECTION = 'LSSTCam/runs/DRP/v30_0_4/DM-54249'   
#COLLECTION = 'LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881' (no skymap)
#COLLECTION = 'LSSTCam/runs/DRP/DP2-pilot/v30_0_4_rc1/DM-54210'
	

#dp2_prep: LSSTCam/runs/DRP/DP2-pilot/v30_0_4_rc1/DM-54210/
# ── Skymap ────────────────────────────────────────────────────────────────────
SKYMAP_NAME = 'lsst_cells_v2'    # nom du skymap pour DP2
INSTRUMENT  = 'LSSTCam'

# ── Bande et champ de référence pour les tests de visualisation ───────────────
BANDS       = ['u', 'g', 'r', 'i', 'z', 'y']
BAND_REF  = 'r'          # bande de référence pour l'exploration

# ECDFS
RA_ECDFS = 53.13 
DEC_ECDFS  = -28.10

# COSMOS
#RA = 150.12° (10h 00m 28.6s)
#Dec = +2.21° (+2° 12′ 21″)
RA_COSMOS = 150.12
DEC_COSMOS = 2.21

# Choose center
FIELD = "COSMOS"
RA_CEN = RA_COSMOS
DEC_CEN = DEC_COSMOS

## 3. Function definition

In [ ]:
def get_plot_path(dataset_type, band, skymap=SKYMAP_NAME, collection=COLLECTION):
    """
    Retourne le chemin POSIX du fichier PNG d'un SurveyWidePropertyMapPlot.

    Parameters
    ----------
    dataset_type : str
        Nom du dataset type Plot.
    band : str
        Bande photométrique (ex: 'r').
    skymap : str
        Nom du skymap (défaut : SKYMAP_NAME).
    collection : str
        Collection butler (défaut : COLLECTION).

    Returns
    -------
    path : str or None
    """
    try:
        uri = butler.getURI(
            dataset_type,
            dataId={'band': band, 'skymap': skymap},
            collections=collection
        )
        return uri.path
    except Exception as e:
        print(f"  Introuvable ({band}) : {type(e).__name__}: {e}")
        return None


def show_plot(dataset_type, band, skymap=SKYMAP_NAME, collection=COLLECTION, figsize=(14, 8)):
    """
    Affiche un SurveyWidePropertyMapPlot pour une bande donnée.
    """
    path = get_plot_path(dataset_type, band, skymap, collection)
    if path is None:
        return
    img = mpimg.imread(path)
    short = (
        dataset_type
        .replace('propertyMapSurvey_deepCoadd_', '')
        .replace('_SurveyWidePropertyMapPlot', '')
    )
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"{short} — band={band}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Fonctions utilitaires définies.")

## 4. Start

## 4. Initalisation

In [ ]:
# Instanciation du Butler (sans collection fixe pour garder la flexibilité des requêtes)
butler   = Butler(REPO)
registry = butler.registry
print(f"Butler instancié sur le dépôt : {REPO}")

In [ ]:
# Retrieve the sky tessellation (skyMap) from the butler.
# The 'lsst_cells_v2' skymap divides the full sky into tracts and patches.
# Error handling is included because the skymap may not be present in all collections.
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME, collections=COLLECTION)
except Exception as inst:
    print(type(inst))   # exception type
    print(inst.args)    # arguments stored in .args
    print(inst)         # full message via __str__

In [ ]:
skymap

## 5. Exploration des collections disponibles dans `dp2_prep`

Avant d'accéder aux cartes, il est utile de lister les collections disponibles  
afin d'identifier celles qui sont susceptibles de contenir des survey property maps.

In [ ]:
# ── Lister TOUTES les collections du dépôt ────────────────────────────────────
all_collections = sorted(registry.queryCollections())
print(f"Nombre total de collections : {len(all_collections)}")
#print("\n".join(all_collections))

In [ ]:
# ── Filtrer les collections potentiellement liées aux survey property maps ────
# Mots-clés à adapter selon les conventions de nommage DP2
keywords = ['survey', 'prop', 'map', 'consolidated', 'spm', 'dp2-prep']

spm_candidates = [
    c for c in all_collections
    if any(kw.lower() in c.lower() for kw in keywords)
]
print(f"Collections candidates ({len(spm_candidates)}) :")
for c in spm_candidates:
    print(" ", c)

In [ ]:
# ── Filtrer les collections DRP de DP2 ────────────────────────────────────────
drp_collections = [
    c for c in all_collections
    if 'DRP' in c or 'drp' in c
]
print(f"Collections DRP ({len(drp_collections)}) :")
for c in drp_collections:
    print(" ", c)

## 6. Recherche des dataset types de type survey property map

Pour DP1, les survey property maps portent le suffixe `consolidated_map`.  
On recherche les types correspondants dans DP2.

In [ ]:
# ── Lister tous les dataset types contenant 'consolidated_map' ────────────────
print("=== Dataset types *consolidated_map* (toutes collections) ===")
for dtype in sorted(registry.queryDatasetTypes(expression="*consolidated_map*")):
    print(" ", dtype.name)

In [ ]:
# ── Recherche plus large : tout ce qui ressemble à une property map ───────────
# Adapter les patterns si les noms DP2 diffèrent de DP1
patterns = [
    '*consolidated_map*',
    '*survey_property*',
    '*property_map*',
    '*healsparse*',
    '*healSparse*',
]

found_types = set()
for pattern in patterns:
    try:
        for dtype in registry.queryDatasetTypes(expression=pattern):
            found_types.add(dtype.name)
    except Exception as e:
        print(f"Pattern '{pattern}' : {e}")

print(f"\nDataset types candidats trouvés ({len(found_types)}) :")
for name in sorted(found_types):
    print(" ", name)

## 7. Separate maps from plots

In [ ]:
type_spmap = []
type_spplot = []
for ftype in found_types:
    if "PropertyMapPlot" in ftype:
        type_spplot.append(ftype)
    elif "deepCoadd_" in ftype and "_map_" in ftype:
        type_spmap.append(ftype) 

In [ ]:
# ── Vérifier lesquels sont présents dans la collection cible ──────────────────
print(f"\nDataset types présents dans la collection : {COLLECTION}")
available_spm = []
for name in sorted(found_types):
    try:
        has_data = registry.queryDatasets(
            name,
            collections=COLLECTION
        ).any(execute=False, exact=False)
        if has_data:
            available_spm.append(name)
            print(f"  ✓ {name}")
    except Exception as e:
        print(f"  ? {name} — {e}")

print(f"\n→ {len(available_spm)} dataset types SPM disponibles dans {COLLECTION}")

In [ ]:
# ── Vérifier lesquels sont présents dans la collection cible ──────────────────
print(f"\nDataset types présents dans la collection : {COLLECTION}")
available_spmap = []
for name in sorted(type_spmap):
    try:
        has_data = registry.queryDatasets(
            name,
            collections=COLLECTION
        ).any(execute=False, exact=False)
        if has_data:
            available_spmap.append(name)
            print(f"  ✓ {name}")
    except Exception as e:
        print(f"  ? {name} — {e}")

print(f"\n→ {len(available_spmap)} dataset types SPM disponibles dans {COLLECTION}")

In [ ]:
# ── Vérifier lesquels sont présents dans la collection cible ──────────────────
print(f"\nDataset types présents dans la collection : {COLLECTION}")
available_spplot = []
for name in sorted(type_spplot):
    try:
        has_data = registry.queryDatasets(
            name,
            collections=COLLECTION
        ).any(execute=False, exact=False)
        if has_data:
            available_spplot.append(name)
            print(f"  ✓ {name}")
    except Exception as e:
        print(f"  ? {name} — {e}")

print(f"\n→ {len(available_spplot)} dataset types SPM disponibles dans {COLLECTION}")

## 8. Inspection des dimensions des dataset types trouvés

Pour DP1 les dimensions requises étaient `{skymap, band}`.  
On vérifie si c'est identique en DP2.

In [ ]:
for name in sorted(available_spm):
    dtype = butler.get_dataset_type(name)
    print(f"{name} :::", dtype)
    print(f"    dimensions requises : {dtype.dimensions.required}")
    print()

## 9. Récupération d'une survey property map

On tente de récupérer la carte de magnitude limite PSF en bande `r`  
(même carte que dans le notebook DP1 de référence).  

**TODO** : remplacer `MAP_NAME` par un nom valide trouvé à la section 4.

In [ ]:
# ── Nom de la carte à récupérer (à adapter) ───────────────────────────────────
# Exemples DP1 (à tester en DP2) :
#   'deepCoadd_psf_maglim_consolidated_map_weighted_mean'
#   'deepCoadd_psf_size_consolidated_map_weighted_mean'
#   'deepCoadd_exposure_time_consolidated_map_sum'
#   'deepCoadd_sky_background_consolidated_map_weighted_mean'

# Utiliser la première carte disponible trouvée à la section 4 si la liste n'est pas vide
if available_spm:
    MAP_NAME = available_spm[0]   # remplacer par le nom souhaité
else:
    MAP_NAME = 'deepCoadd_psf_maglim_consolidated_map_weighted_mean'  # à adapter

print(f"Carte sélectionnée : {MAP_NAME}")
print(f"Bande              : {BAND_REF}")
print(f"Collection         : {COLLECTION}")

In [ ]:
uri  = butler.getURI(
    MAP_NAME,
    dataId={'band': BAND_REF, 'skymap': SKYMAP_NAME},
    collections=COLLECTION
)
path = uri.path
print(f"URI  : {uri}")
print(f"Path : {path}")
print(f"Ext  : {Path(path).suffix}")

In [ ]:
# Lecture et affichage
img = mpimg.imread(path)
print(f"Shape de l'image : {img.shape}")

fig, ax = plt.subplots(figsize=(18, 8))
ax.imshow(img)
ax.axis('off')
short = MAP_NAME.replace('propertyMapSurvey_deepCoadd_', '').replace('_SurveyWidePropertyMapPlot', '')
ax.set_title(f"{short} — band={BAND_REF}", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Récupération de la carte ───────────────────────────────────────────────────
try:
    hspmap = butler.get(
        MAP_NAME,
        dataId = {"band":BAND_REF,"skymap" : SKYMAP_NAME} ,
        collections=COLLECTION
    )
    print(f"Carte récupérée avec succès : {type(hspmap)}")
    print(hspmap)
except Exception as e:
    print(f"Erreur lors de la récupération : {type(e).__name__}: {e}")
    hspmap = None

In [ ]:
# Les 14 dataset types Plot confirmés dans la collection
PLOT_DATASET_TYPES = [
    'propertyMapSurvey_deepCoadd_dcr_ddec_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_dra_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_e1_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_dcr_e2_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_max_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_epoch_consolidated_map_min_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_exposure_time_consolidated_map_sum_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_e1_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_e2_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_maglim_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_psf_size_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_sky_background_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
    'propertyMapSurvey_deepCoadd_sky_noise_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot',
]
rows = []
for name in PLOT_DATASET_TYPES:
    short = (
        name
        .replace('propertyMapSurvey_deepCoadd_', '')
        .replace('_SurveyWidePropertyMapPlot', '')
    )
    for band in BANDS:
        path = get_plot_path(name, band)
        rows.append({
            'dataset_type': name,
            'map': short,
            'band': band,
            'path': path,
            'ok': path is not None
        })

df = pd.DataFrame(rows)

# Tableau récapitulatif disponibilité
pivot = df.pivot(index='map', columns='band', values='ok')
print(f"Plots disponibles (True = fichier trouvé) :")
print(pivot.to_string())
print(f"\nTotal disponibles : {df['ok'].sum()} / {len(df)}")

## 10. Afficher un plot par bande — une carte au choix

In [ ]:
# Changer PLOT_NAME pour explorer une autre carte
#PLOT_NAME = 'propertyMapSurvey_deepCoadd_psf_maglim_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot'
PLOT_NAME = 'propertyMapSurvey_deepCoadd_psf_maglim_consolidated_map_weighted_mean_SurveyWidePropertyMapPlot'
df_map = df[(df['dataset_type'] == PLOT_NAME) & df['ok']]
n = len(df_map)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3 * nrows),layout="constrained")
axes = np.array(axes).flatten()

for idx, (_, row) in enumerate(df_map.iterrows()):
    img = mpimg.imread(row['path'])
    axes[idx].imshow(img)
    axes[idx].axis('off')
    axes[idx].set_title(f"band = {row['band']}", fontsize=13, fontweight='bold')

for idx in range(n, len(axes)):
    axes[idx].set_visible(False)

short = PLOT_NAME.replace('propertyMapSurvey_deepCoadd_', '').replace('_SurveyWidePropertyMapPlot', '')
plt.suptitle(short, fontsize=14, fontweight='bold', y=1.01)
#plt.tight_layout()
plt.show()

## 11. Affichage carte par carte, toutes bandes

Pour chaque type de carte, une figure avec les 6 bandes côte à côte.

In [ ]:
for map_name in df['map'].unique():
    df_sub = df[(df['map'] == map_name) & df['ok']]
    if df_sub.empty:
        continue

    n = len(df_sub)
    ncols = min(n, 3)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3 * nrows),layout="constrained")
    axes = np.array(axes).flatten()

    for idx, (_, row) in enumerate(df_sub.iterrows()):
        img = mpimg.imread(row['path'])
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(f"band = {row['band']}", fontsize=11, fontweight='bold')

    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle(map_name, fontsize=11, fontweight='bold')
    #plt.tight_layout()
    plt.show()

## 11. Galerie des 14 cartes — bande r

In [ ]:
df_r = df[(df['band'] == BAND_REF) & df['ok']]

ncols = 2
nrows = int(np.ceil(len(df_r) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows),layout="constrained")
axes = np.array(axes).flatten()

for idx, (_, row) in enumerate(df_r.iterrows()):
    img = mpimg.imread(row['path'])
    axes[idx].imshow(img)
    axes[idx].axis('off')
    axes[idx].set_title(row['map'], fontsize=8)

for idx in range(len(df_r), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(
    f"Survey Property Map Plots — DP2 — band={BAND_REF}\n{COLLECTION}",
    fontsize=13, fontweight='bold', y=1.005
)
#plt.tight_layout()
plt.show()

## 12. Recherche des HealSparseMap brutes dans d'autres collections

Les plots sont disponibles mais les objets `HealSparseMap` interrogeables  
(noms sans `propertyMapSurvey_` et sans `_SurveyWidePropertyMapPlot`)  
sont peut-être dans une collection différente.

In [ ]:
# Noms des HealSparseMap correspondantes
raw_map_names = [
    name
    .replace('propertyMapSurvey_', '')
    .replace('_SurveyWidePropertyMapPlot', '')
    for name in PLOT_DATASET_TYPES
]

# Chercher dans toutes les collections du dépôt
all_collections = sorted(registry.queryCollections())
MAP_TO_SEARCH   = raw_map_names[7]  # exposure_time — souvent disponible

print(f"Recherche de '{MAP_TO_SEARCH}'\ndans toutes les collections du dépôt {REPO} :\n")
found_in = []
for coll in all_collections:
    try:
        has = registry.queryDatasets(
            MAP_TO_SEARCH, collections=coll
        ).any(execute=False, exact=False)
        if has:
            found_in.append(coll)
            print(f"  ✓  {coll}")
    except Exception:
        pass

if not found_in:
    print("  Aucune collection ne contient les HealSparseMap brutes.")
    print("  Seuls les plots PNG sont disponibles dans ce dépôt.")
   

In [ ]:
# Si des collections ont été trouvées, récupérer une HealSparseMap
hspmap = None
if found_in:
    coll_hsp = found_in[0]
    #coll_hsp = COLLECTION
    print(f"Récupération dans : {coll_hsp}")
    try:
        hspmap = butler.get(
            MAP_TO_SEARCH,
            dataId={'band': BAND_REF, 'skymap': SKYMAP_NAME},
            collections=coll_hsp
        )
        print(f"  type : {type(hspmap)}")
        print(hspmap)
        val = hspmap.get_values_pos(RA_CEN, DEC_CEN)
        print(f"  Values in selected field {FIELD} : {val}")
    except Exception as e:
        print(f"  Erreur : {type(e).__name__}: {e}")
else:
    print("Aucune HealSparseMap disponible — utiliser les plots PNG (sections 3–8).")

## 7. Valeur de la carte à un point de référence

On interroge la valeur de la carte au centre du champ FIELD 
(même position de référence que dans le notebook DP1).

In [ ]:
if hspmap is not None:
    val_center = hspmap.get_values_pos(RA_CEN, DEC_CEN)
    print(f"Valeur de '{MAP_NAME}' ({BAND_REF}-band) au centre {FIELD}")
    print(f"  RA={RA_CEN}, Dec={DEC_CEN}  →  {val_center}")

    # Sentinelle : valeur hors couverture
    val_north = hspmap.get_values_pos(180.0, +60.0)
    print(f"  Valeur sentinelle (hors couverture, RA=180, Dec=+60) → {val_north}")
else:
    print("Carte non disponible — section ignorée.")

## 8. Statistiques sur une région autour du FIELD

In [ ]:
if hspmap is not None:
    span_dec = 2.0
    span_ra  = span_dec / np.cos(np.deg2rad(DEC_CEN))

    ra  = np.linspace(RA_CEN  - span_ra,  RA_CEN  + span_ra,  250)
    dec = np.linspace(DEC_CEN - span_dec, DEC_CEN + span_dec, 250)
    x, y = np.meshgrid(ra, dec)

    values = hspmap.get_values_pos(x, y)

    # Remplacer la valeur sentinelle par NaN
    sentinel = hspmap.get_values_pos(180.0, +60.0)
    values[values == sentinel] = np.nan

    print(f"Région {FIELD} — {MAP_NAME} ({BAND_REF}-band)")
    print(f"  min    = {np.nanmin(values):.4f}")
    print(f"  max    = {np.nanmax(values):.4f}")
    print(f"  mean   = {np.nanmean(values):.4f}")
    print(f"  median = {np.nanmedian(values):.4f}")
    print(f"  std    = {np.nanstd(values):.4f}")
else:
    print("Carte non disponible — section ignorée.")

## 9. Visualisation locale (pcolormesh)

Carte de la propriété sélectionnée dans la région ECDFS,  
en utilisant `pcolormesh` (identique au notebook DP1 de référence).

In [ ]:
if hspmap is not None:
    fig, ax = plt.subplots(figsize=(7, 6))
    pcm = ax.pcolormesh(x, y, values, cmap='viridis')
    fig.colorbar(pcm, ax=ax, label=f"{MAP_NAME.split('_')[-2]} ({BAND_REF}-band)")
    ax.set_xlabel("Right Ascension (deg)")
    ax.set_ylabel("Declination (deg)")
    ax.set_title(f"{MAP_NAME}\n{BAND_REF}-band — {FIELD}", fontsize=11)
    ax.invert_xaxis()   # convention : Est à gauche
    plt.tight_layout()
    plt.show()
else:
    print(f"Carte non disponible in field {FIELD} — visualisation ignorée.")

## 10. Visualisation all-sky (skyproj)

Vue globale de la carte sur l'ensemble du ciel avec `skyproj`,  
si le paquet est disponible dans l'environnement.

In [ ]:
if hspmap is not None and HAS_SKYPROJ:
    # Projection McBryde centrée sur le champ de référence
    fig, ax = plt.subplots(figsize=(10, 5))
    sp = skyproj.McBrydeSkyproj(ax=ax, lon_0=RA_CEN)
    sp.draw_hspmap(hspmap)
    sp.draw_colorbar(label=f"{MAP_NAME} ({BAND_REF}-band)")
    ax.set_title(f"All-sky — {MAP_NAME} ({BAND_REF}-band)", fontsize=12)
    #plt.tight_layout()
    plt.show()
elif hspmap is not None:
    print("skyproj non disponible — visualisation all-sky ignorée.")
else:
    print("Carte non disponible — visualisation ignorée.")

## 11. Boucle sur plusieurs cartes disponibles

Récupérer et afficher toutes les survey property maps disponibles en bande `r`  
dans la collection cible.

In [ ]:
if not available_spm:
    print("Aucun dataset type SPM trouvé — section ignorée.")
else:
    n_maps = len(available_spm)
    print(f"Tentative de récupération de {n_maps} cartes en bande '{BAND_REF}'...\n")

    maps = {}
    for name in available_spm:
        try:
            m = butler.get(name, dataId={'band': BAND_REF, 'skymap': SKYMAP_NAME}, collections=coll_hsp)
            maps[name] = m
            print(f"  ✓ {name}")
        except Exception as e:
            print(f"  ✗ {name}  — {type(e).__name__}: {e}")

    print(f"\n{len(maps)}/{n_maps} cartes récupérées avec succès.")

In [ ]:
# ── Affichage en grille de toutes les cartes récupérées ───────────────────────
if maps:
    n_ok = len(maps)
    ncols = 2
    nrows = int(np.ceil(n_ok / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4.5 * nrows))
    axes = np.array(axes).flatten()

    span_dec = 0.75
    span_ra  = span_dec / np.cos(np.deg2rad(DEC_CEN))
    ra_arr   = np.linspace(RA_CEN  - span_ra,  RA_CEN  + span_ra,  200)
    dec_arr  = np.linspace(DEC_CEN - span_dec, DEC_CEN + span_dec, 200)
    xg, yg   = np.meshgrid(ra_arr, dec_arr)

    for idx, (name, hsp) in enumerate(maps.items()):
        ax = axes[idx]
        vals = hsp.get_values_pos(xg, yg).astype(float)
        # Sentinelle → NaN
        sentinel = hsp.get_values_pos(180.0, +60.0)
        vals[vals == sentinel] = np.nan

        pcm = ax.pcolormesh(xg, yg, vals, cmap='viridis')
        fig.colorbar(pcm, ax=ax)
        short_name = name.replace('deepCoadd_', '').replace('_consolidated_map_', ' ')
        ax.set_title(f"{short_name}\n({BAND_REF}-band)", fontsize=9)
        ax.set_xlabel("RA (deg)")
        ax.set_ylabel("Dec (deg)")
        ax.invert_xaxis()

    # Masquer les axes vides
    for idx in range(n_ok, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle(f"Survey Property Maps DP2 — région ECDFS — bande {BAND_REF}",
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

## 12. Récupération sans précision de la bande (cartes band-independantes)

Certaines cartes (e.g. `exposure_time`, `epoch`) peuvent ne pas nécessiter  
le paramètre `band`. On les explore ici.

In [ ]:
# ── Tentative de récupération sans band (skymap only) ─────────────────────────
if not available_spm:
    print("Aucun dataset type SPM trouvé — section ignorée.")
else:
    print("Tentative de récupération des cartes sans paramètre 'band' :")
    for name in available_spm:
        try:
            m = butler.get(name, collections=COLLECTION)
            print(f"  ✓ {name}  (band-independent)")
        except TypeError as e:
            print(f"  — {name}  nécessite 'band' : {e}")
        except Exception as e:
            print(f"  ✗ {name}  : {type(e).__name__}: {e}")

## 13. Dump complet des dataset types disponibles dans la collection cible

Liste exhaustive de tous les dataset types présents dans `COLLECTION_SPMAPS`,  
en excluant les produits de bookkeeping (configs, logs, metrics).

In [ ]:
EXCLUDE_SUFFIXES = (
    '_config', '_log', '_metadata', '_resource_usage',
    'Plot', 'Metric', 'metric'
)

print(f"Dataset types présents dans '{COLLECTION}' (hors bookkeeping) :")
science_dtypes = []
for dt in registry.queryDatasetTypes():
    if any(s in dt.name for s in EXCLUDE_SUFFIXES):
        continue
    try:
        if registry.queryDatasets(
            dt, collections=COLLECTION
        ).any(execute=False, exact=False):
            science_dtypes.append(dt.name)
    except Exception:
        pass

science_dtypes.sort()
for name in science_dtypes:
    print(" ", name)

print(f"\nTotal : {len(science_dtypes)} dataset types")

## 14. Notes et prochaines étapes

- Mettre à jour `COLLECTION` avec la collection DP2 correcte après avoir inspecté la sortie de la **section 3**.
- Si les noms de dataset types diffèrent de DP1 (e.g. `deepCoadd_*` → `step3_*` ou autre), adapter les patterns de recherche à la **section 4**.
- Une fois les bonnes cartes identifiées, ce notebook peut être enrichi avec :
  - Comparaison DP1 vs DP2 pour les mêmes champs.
  - Analyse de la couverture spatiale par bande.
  - Histogrammes de distribution des valeurs des cartes.